[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/recipes/blob/main/weaviate-features/model-providers/digitalocean/digitalocean_rerank.ipynb)

# Retrieve, rerank and generate with DigitalOcean Serverless Inference

This notebook builds a small RAG pipeline that runs entirely on [DigitalOcean Serverless Inference](https://docs.digitalocean.com/products/gradient-ai-platform/how-to/use-serverless-inference/):

1. **Embed** with `text2vec-digitalocean` (`bge-m3`)
2. **Rerank** the candidates with `reranker-digitalocean` (`bge-reranker-v2-m3`, a cross-encoder)
3. **Generate** an answer with `generative-digitalocean`

Vector search is fast but approximate: it finds passages that are *about* the same thing as the query. A cross-encoder reranker reads the query and each candidate together, so it is much better at telling which passage actually *answers* the question.

> **Prerequisites:** a running Weaviate with the three DigitalOcean modules enabled, and a DigitalOcean model access key in `DIGITALOCEAN_APIKEY`. See [README.md](./README.md).

## Dependencies

In [ ]:
%pip install -U -q weaviate-client

## Connect to Weaviate

The key is sent with each request in the `X-Digitalocean-Api-Key` header, so Weaviate does not need it in its environment.

In [ ]:
import os
import weaviate

digitalocean_key = os.environ["DIGITALOCEAN_APIKEY"]

client = weaviate.connect_to_local(
    headers={"X-Digitalocean-Api-Key": digitalocean_key},
)

modules = client.get_meta()["modules"]
for name in ("text2vec-digitalocean", "generative-digitalocean", "reranker-digitalocean"):
    assert name in modules, f"{name} is not enabled in this Weaviate instance"
print("Weaviate", client.get_meta()["version"], "- DigitalOcean modules enabled")

## Create a collection

`reranker-digitalocean` is new, so the Python client has no dedicated helper for it yet. `Configure.Reranker.custom` takes the module name and its settings:

| Setting | Default |
|---|---|
| `model` | `bge-reranker-v2-m3` |
| `baseURL` | `https://inference.do-ai.run` |

In [ ]:
import weaviate.classes.config as wc

COLLECTION = "BakingTips"

if client.collections.exists(COLLECTION):
    client.collections.delete(COLLECTION)

tips = client.collections.create(
    name=COLLECTION,
    vector_config=wc.Configure.Vectors.text2vec_digitalocean(
        model="bge-m3",
        source_properties=["title", "content"],
    ),
    reranker_config=wc.Configure.Reranker.custom(
        "reranker-digitalocean",
        module_config={"model": "bge-reranker-v2-m3"},
    ),
    generative_config=wc.Configure.Generative.digitalocean(
        model="openai-gpt-5-mini",
    ),
    properties=[
        wc.Property(name="title", data_type=wc.DataType.TEXT),
        wc.Property(name="content", data_type=wc.DataType.TEXT),
    ],
)
print(f"Created collection {COLLECTION}")

## Import data

The passages are deliberately close to each other: several talk about sourdough, yeast or starters, but only a few actually answer the questions asked below.

In [ ]:
passages = [
    ("Feeding a sourdough starter", "To keep a sourdough starter alive, discard all but about 50 g and feed it equal weights of flour and water every 12 to 24 hours at room temperature. Feed it once a week if it lives in the fridge."),
    ("Reviving a neglected starter", "A starter left unfed for weeks develops a grey liquid called hooch. Pour it off, then feed the starter twice a day for two or three days until it doubles reliably again."),
    ("History of sourdough", "Sourdough is one of the oldest forms of leavened bread, dating back to ancient Egypt around 1500 BC. San Francisco sourdough became famous during the California Gold Rush."),
    ("Sourdough vs commercial yeast", "Sourdough relies on wild yeast and lactic acid bacteria, which give it a tangy flavour. Commercial yeast is a single strain that ferments faster and more predictably."),
    ("Proofing dough in the fridge", "Cold retarding dough overnight in the fridge slows fermentation, develops flavour and makes the dough easier to score before baking."),
    ("Why bread comes out dense", "Dense bread is usually under-proofed: the dough did not ferment long enough to trap gas. A weak starter, cold kitchen or too little water can all cause it."),
    ("Measuring flour", "Spooning flour into a cup and levelling it gives about 120 g per cup. Scooping directly with the cup packs the flour and can add 20 percent more."),
    ("Choosing bread flour", "Bread flour has more protein than all-purpose flour, typically 12 to 14 percent, which builds stronger gluten and gives a chewier crumb."),
    ("Instant vs active dry yeast", "Instant yeast can be mixed straight into the flour. Active dry yeast should be dissolved in warm water first. Use about 25 percent less instant yeast than active dry."),
    ("Storing yeast", "Unopened packets of dried yeast keep for up to two years in a cool cupboard. Once opened, store yeast in an airtight container in the fridge or freezer."),
    ("Baking with steam", "Steam in the first 15 minutes of baking keeps the crust soft so the loaf can expand fully. A Dutch oven traps the dough's own steam and is the easiest way to get it at home."),
    ("Cooling bread before slicing", "Bread keeps cooking as it cools. Slicing a loaf straight from the oven releases steam and leaves the crumb gummy, so wait at least an hour."),
    ("Making pizza dough", "A simple pizza dough uses flour, water, salt and a little yeast. A long cold ferment of two or three days gives a lighter, more flavourful crust."),
    ("Gluten-free baking", "Gluten-free breads need binders such as psyllium husk or xanthan gum to hold gas, since there is no gluten network to trap it."),
    ("Kitchen scale basics", "Baking by weight is far more consistent than by volume. A digital scale that reads in 1 g steps is enough for almost every bread recipe."),
]

with tips.batch.fixed_size(batch_size=50) as batch:
    for title, content in passages:
        batch.add_object({"title": title, "content": content})

failed = tips.batch.failed_objects
assert not failed, failed[0].message
print(f"Imported {len(tips)} passages")

## Search without reranking

In [ ]:
QUERY = "how often should I feed my starter so it stays alive?"

def show(response):
    for i, obj in enumerate(response.objects, 1):
        score = obj.metadata.rerank_score
        score = f"{score:8.3f}" if score is not None else "       -"
        print(f"{i}. rerank={score}  {obj.properties['title']}")

response = tips.query.near_text(QUERY, limit=6)
show(response)

## Search with reranking

`limit` sets how many candidates vector search retrieves. The reranker then scores every candidate against `query` using the `content` property, and Weaviate returns them sorted by that score.

The scores are raw cross-encoder logits, so they can be negative. Only their order matters.

In [ ]:
import weaviate.classes.query as wq

response = tips.query.near_text(
    QUERY,
    limit=6,
    rerank=wq.Rerank(prop="content", query=QUERY),
)
show(response)

## Hybrid search with reranking

Reranking works with any search type. Here it reorders hybrid (BM25 + vector) results for a question whose keywords appear in several passages.

In [ ]:
question = "my loaf is heavy and dense, what went wrong?"

response = tips.query.hybrid(
    question,
    limit=6,
    rerank=wq.Rerank(prop="content", query=question),
)
show(response)

## Retrieve, rerank and generate

With reranking, the passages handed to the language model are the most relevant ones, not just the closest vectors.

In [ ]:
response = tips.generate.near_text(
    QUERY,
    limit=6,
    rerank=wq.Rerank(prop="content", query=QUERY),
    grouped_task=f"Using only these passages, answer in two sentences: {QUERY}",
)
show(response)
print()
print(response.generative.text)

## Clean up

In [ ]:
client.collections.delete(COLLECTION)
client.close()